In [1]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Lambda
)

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

2026-06-28 15:13:05.035345: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782659585.215166      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782659585.266607      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782659585.685582      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782659585.685628      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782659585.685631      24 computation_placer.cc:177] computation placer alr

In [2]:
CSV_PATH = "/kaggle/input/notebooks/sumedhabhadauria/01-dataset-preparation/clean_metadata.csv"
MODEL_PATH = "/kaggle/input/notebooks/sumedhabhadauria/03-resnet50-feature-extraction/feature_extractor.keras"
IMAGE_FOLDER = Path("/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset/images")

In [3]:
df = pd.read_csv(CSV_PATH)
feature_extractor = load_model(MODEL_PATH)
print("Total Products :", len(df))

I0000 00:00:1782659598.708393      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782659598.714415      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Total Products : 44419


In [4]:
# Creating a Balanced Training Subset

TOP_CLASSES = 10
SAMPLES_PER_CLASS = 500
top_article_types = (
    df["articleType"]
    .value_counts()
    .head(TOP_CLASSES)
    .index
)
subset = []
for article in top_article_types:
    temp = df[df["articleType"] == article]
    temp = temp.sample(
        min(SAMPLES_PER_CLASS, len(temp)),
        random_state=42
    )
    subset.append(temp)
subset_df = (
    pd.concat(subset)
    .reset_index(drop=True)
)
print("Subset Size :", len(subset_df))
subset_df.head()

Subset Size : 5000


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,24050,Men,Apparel,Topwear,Tshirts,Blue,Fall,2011,Casual,Locomotive Men Printed Blue T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...
1,13967,Men,Apparel,Topwear,Tshirts,Red,Fall,2011,Casual,Ed Hardy Men Printed Red Tshirts,/kaggle/input/datasets/paramaggarwal/fashion-p...
2,4256,Men,Apparel,Topwear,Tshirts,Grey Melange,Summer,2011,Casual,Inkfruit Men Grey Melange Printed T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...
3,50194,Boys,Apparel,Topwear,Tshirts,Black,Summer,2012,Casual,Gini and Jony Boys Black T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...
4,58372,Men,Apparel,Topwear,Tshirts,Beige,Summer,2012,Casual,Locomotive Men Beige T-shirt,/kaggle/input/datasets/paramaggarwal/fashion-p...


In [5]:
positive_pairs = []
for article in subset_df["articleType"].unique():
    temp = subset_df[subset_df["articleType"] == article]
    image_list = temp["id"].tolist()

    for i in range(len(image_list) - 1):
        positive_pairs.append(
            (
                image_list[i],
                image_list[i + 1],
                1
            )
        )
print("Positive Pairs :", len(positive_pairs))

Positive Pairs : 4990


In [6]:
negative_pairs = []
article_types = subset_df["articleType"].unique()

for article in article_types:
    current = subset_df[
        subset_df["articleType"] == article
    ]["id"].tolist()
    other = subset_df[
        subset_df["articleType"] != article
    ]["id"].tolist()
    sample_size = min(len(current), len(other))
    random_other = random.sample(
        other,
        sample_size
    )

    for img1, img2 in zip(current, random_other):
        negative_pairs.append(
            (
                img1,
                img2,
                0
            )
        )
print("Negative Pairs :", len(negative_pairs))

Negative Pairs : 5000


In [7]:
pairs = positive_pairs + negative_pairs
random.shuffle(pairs)
pairs_df = pd.DataFrame(
    pairs,
    columns=[
        "image1",
        "image2",
        "label"
    ]
)
print("Total Pairs :", len(pairs_df))
pairs_df.head()

Total Pairs : 9990


,image1,image2,label
0,17844,47361,0
1,7207,22388,0
2,27534,2101,1
3,41967,15604,1
4,46508,27262,0


In [8]:
train_df, test_df = train_test_split(
    pairs_df,
    test_size=0.2,
    random_state=42,
    stratify=pairs_df["label"]
)

print("Training Pairs :", len(train_df))
print("Testing Pairs :", len(test_df))

Training Pairs : 7992
Testing Pairs : 1998


In [9]:
def load_image(image_id):
    img_path = IMAGE_FOLDER / f"{image_id}.jpg"
    img = image.load_img(
        img_path,
        target_size=(224,224)
    )
    img = image.img_to_array(img)
    img = preprocess_input(img)
    return img

In [10]:
# Creating Training Arrays

X1 = []
X2 = []
Y = []

for _, row in train_df.iterrows():
    try:
        img1 = load_image(row["image1"])
        img2 = load_image(row["image2"])
        X1.append(img1)
        X2.append(img2)
        Y.append(row["label"])
    except:
        continue

X1 = np.array(X1)
X2 = np.array(X2)
Y = np.array(Y)
print("Training Samples :", len(Y))

Training Samples : 7992


In [11]:
# Building the Siamese Network

base_model = feature_extractor
base_model.trainable = False

input_a = Input(shape=(224,224,3))
input_b = Input(shape=(224,224,3))

embedding_a = base_model(input_a)
embedding_b = base_model(input_b)

distance = Lambda(
    lambda tensors: tf.abs(
        tensors[0] - tensors[1]
    )
)([embedding_a, embedding_b])

x = Dense(
    512,
    activation="relu"
)(distance)

x = Dense(
    128,
    activation="relu"
)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)

siamese_model = Model(
    [input_a,input_b],
    output
)

siamese_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 2048)      │ 23,587,712 │ input_layer[0][0… │
│ (Functional)        │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 2048)      │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,049,088 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     65,664 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        129 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,702,593 (94.23 MB)

 Trainable params: 1,114,881 (4.25 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [12]:
# Training

siamese_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = siamese_model.fit(
    [X1,X2],
    Y,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5


I0000 00:00:1782659982.954841      69 service.cc:152] XLA service 0x78ceb01155f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782659982.954905      69 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782659982.954913      69 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782659986.705180      69 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/200 ━━━━━━━━━━━━━━━━━━━━ 1:09:13 21s/step - accuracy: 0.3125 - loss: 4.4317

I0000 00:00:1782659991.791056      69 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 311ms/step - accuracy: 0.8042 - loss: 0.7146 - val_accuracy: 0.8718 - val_loss: 0.3710
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 42s 208ms/step - accuracy: 0.8918 - loss: 0.2723 - val_accuracy: 0.9018 - val_loss: 0.2607
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.9043 - loss: 0.2386 - val_accuracy: 0.8999 - val_loss: 0.2886
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 205ms/step - accuracy: 0.9140 - loss: 0.2146 - val_accuracy: 0.8937 - val_loss: 0.3369
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.9295 - loss: 0.1780 - val_accuracy: 0.8981 - val_loss: 0.3471


In [13]:
# Saving the Model

siamese_model.save("/kaggle/working/siamese_model.keras")
print("Siamese Model Saved Successfully")

Siamese Model Saved Successfully


In [14]:
import pickle
with open("/kaggle/working/history.pkl","wb") as f:
    pickle.dump(
        history.history,
        f
    )
print("Training History Saved Successfully")

Training History Saved Successfully


In [15]:
print("Training Samples      :",len(train_df))
print("Testing Samples       :",len(test_df))
print("Final Train Accuracy  :",
      round(history.history["accuracy"][-1]*100,2),
      "%")
print("Final Validation Accuracy :",
      round(history.history["val_accuracy"][-1]*100,2),
      "%")

Training Samples      : 7992
Testing Samples       : 1998
Final Train Accuracy  : 92.95 %
Final Validation Accuracy : 89.81 %
